# Week 3 · Module 2 Practical — Five Patterns, Five Builds 🧩

**PMS RoBoTics Research Center · 4-Week Internship & Training Program**
**Week 3 · Module 2: Workflow patterns — the controlled shapes**

---

| # | Pattern | You build |
|---|---------|-----------|
| 1 | **Chaining** | Product-copy pipeline with a gate that FAILS (on purpose) |
| 2 | **Routing** | Support router: 1 classifier + 3 specialist prompts |
| 3 | **Parallelization** | 3-review fan-out (sectioning) + 3-vote majority (voting) |
| 4 | **Orchestrator–workers** | LLM plans a buying guide; workers execute; two call TOOLS |
| 5 | **Evaluator–optimizer** | Draft ⇄ judge loop until PASS |
| 🏁 | **Compose** | SmartSupport pipeline: route → handle → evaluate, with a call counter |

> 📏 A global `CALLS` counter tracks every LLM call — each pattern prints its price. Slide 12's table, verified live.

## Step 0 — Setup 🔑

In [ ]:
%pip install -q -U openai

In [ ]:
import json
from getpass import getpass
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI

API_KEY = getpass("Paste the OpenAI API key: ")
client = OpenAI(api_key=API_KEY)
MODEL = "gpt-4o-mini"

CALLS = {"n": 0}                       # the price tag on every pattern

def ask(prompt, temperature=0.3, max_tokens=400):
    CALLS["n"] += 1
    r = client.chat.completions.create(model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content.strip()

def price_tag(label):
    print(f"   💰 {label}: {CALLS['n']} LLM calls so far")

# Yesterday's tools (condensed) — the orchestrator's workers will use these
PRICES = {"P-101": 1299, "P-102": 1699, "P-103": 899, "P-104": 2499}
NAMES  = {"P-101": "Prima Electric Kettle 1.5L", "P-102": "Prima Electric Kettle 2.0L",
          "P-103": "Zenith Steel Kettle 1.0L",   "P-104": "Zenith Pro Kettle 1.8L"}
STOCK  = {"P-101": 42, "P-102": 18, "P-103": 65, "P-104": 7}

def get_price(product_id):
    """Price in rupees for a product ID like P-101."""
    return f"{NAMES[product_id]}: Rs. {PRICES[product_id]:,}" if product_id in PRICES else "unknown ID"

def check_inventory(product_id):
    """Units in stock for a product ID like P-101."""
    return f"{STOCK[product_id]} units" if product_id in STOCK else "unknown ID"

TOOLS = {"get_price": get_price, "check_inventory": check_inventory}
print("✅ Ready. Call counter armed.")

---
## Pattern 1 — Prompt chaining ⛓️

Three focused calls + one **gate** (plain Python, zero cost). Each prompt does ONE thing — compare with cramming everything into one mega-prompt:

In [ ]:
RAW_PRODUCT = """prima kettle new model 1.5 liter steel body 1500watt boils fast has auto
cutoff when boiling and dry boil protection also 1 yr warranty mrp 1299 color silver"""

EXTRACT = "Extract the product specs from this messy note as a clean bullet list, facts only:\n{raw}"
DRAFT   = "Write a warm 40-word website description using ONLY these specs:\n{specs}"
POLISH  = "Tighten this to crisp, professional website copy. Keep under 45 words, no exclamation marks:\n{draft}"

CALLS["n"] = 0
specs = ask(EXTRACT.format(raw=RAW_PRODUCT), temperature=0)
print("STEP 1 · specs:\n", specs[:200], "\n")

draft = ask(DRAFT.format(specs=specs), temperature=0.8)
print("STEP 2 · draft:\n", draft, "\n")

# ---- THE GATE: plain code between LLM calls ----
wc = len(draft.split())
print(f"GATE   · word count = {wc} → {'PASS' if wc <= 45 else 'FAIL — retrying step 2 with stricter prompt'}")
if wc > 45:
    draft = ask(DRAFT.format(specs=specs) + "\nHARD LIMIT: 40 words maximum.", temperature=0.4)
    print(f"RETRY  · word count = {len(draft.split())}\n")

final = ask(POLISH.format(draft=draft), temperature=0.3)
print("STEP 3 · final:\n", final)
price_tag("chain")

**Why the gate matters:** it's free (no LLM), deterministic, and catches drift *between* steps instead of at the end. Gates can be word counts, `json.loads`, keyword checks, regex — anything Python can verify.

**✏️ Your turn:** add a second gate after POLISH — reject the copy if it contains the word "amazing" (LLM cliché #1), retry once.

---
## Pattern 2 — Routing 🔀

One cheap classifier picks the lane; each lane gets a **specialist prompt** (and could get its own model/temperature). You built this in W1M4 — now it has a name:

In [ ]:
CLASSIFY = """Classify into exactly one: COMPLAINT, REFUND, PRODUCT, OTHER.
Reply with the label only.
Message: "{msg}"
"""

ROUTES = {
    "COMPLAINT": """You are SmartMart's senior support specialist. The customer is upset.
FIRST acknowledge their frustration, then give ONE concrete next step, then offer escalation
to pmssupport@smartmart.example. Max 3 sentences. Message: "{msg}"
""",
    "REFUND": """You are SmartMart's refunds desk. State the policy precisely: 7 days with
receipt = full refund; no receipt = store credit. Ask for their order number. Max 3 sentences.
Message: "{msg}"
""",
    "PRODUCT": """You are SmartMart's friendly product expert. Answer enthusiastically but
honestly; if you don't know a spec, say you'll check. Max 3 sentences. Message: "{msg}"
""",
    "OTHER": """You are SmartMart's assistant. Politely explain you can only help with store,
order and product questions. One sentence. Message: "{msg}"
""",
}

def route_and_reply(msg):
    route = ask(CLASSIFY.format(msg=msg), temperature=0)
    route = route if route in ROUTES else "OTHER"          # gate on the classifier!
    return route, ask(ROUTES[route].format(msg=msg), temperature=0.6)

CALLS["n"] = 0
for msg in ["my kettle died after 2 days and nobody replies to my emails!!",
            "want my money back for order 4521",
            "does the 2L kettle have keep-warm mode?",
            "can you do my maths homework"]:
    route, reply = route_and_reply(msg)
    print(f"[{route}] {msg[:45]}")
    print(f"   → {reply}\n")
price_tag("routing (4 messages)")

**Note the two moves:** the fallback gate (`route if route in ROUTES else "OTHER"`) — never trust the classifier blindly — and the `ROUTES` dict, which is yesterday's `TOOLS` dict pattern again. Dispatch-on-a-dict is the recurring move of this whole week.

**✏️ Your turn:** add a `HINGLISH` route with a prompt that replies in the same language mix. Which existing route steals its traffic before you add it?

---
## Pattern 3 — Parallelization ⚡

**Sectioning** (split the work) and **voting** (repeat the same judgment). `ThreadPoolExecutor` runs the calls concurrently — N calls, ~1× latency:

In [ ]:
import time

REVIEWS = [
    "Bought the Prima kettle last month. Boils fast, handle stays cool. The lid is a bit stiff though.",
    "kettle stopped working in 10 days, service center took 3 weeks. very disappointed with after sales.",
    "Value for money! Auto cutoff works great, my chai routine is 2x faster now. Highly recommend.",
]
SUMMARIZE = "Summarize this product review in ONE line (sentiment + main point): {r}"

# --- serial, timed ---
CALLS["n"] = 0
t0 = time.perf_counter()
serial = [ask(SUMMARIZE.format(r=r)) for r in REVIEWS]
t_serial = time.perf_counter() - t0

# --- parallel, timed ---
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=3) as ex:
    parallel = list(ex.map(lambda r: ask(SUMMARIZE.format(r=r)), REVIEWS))
t_parallel = time.perf_counter() - t0

for s in parallel: print("•", s)
print(f"\n⏱ serial: {t_serial:.1f}s  vs  parallel: {t_parallel:.1f}s  ({t_serial/t_parallel:.1f}x faster)")

# fan-in: aggregate the sections
overview = ask("Combine these one-line review summaries into a 2-sentence overview for the product page:\n"
               + "\n".join(parallel))
print("\nFAN-IN →", overview)
price_tag("sectioning (3 + 1 calls)")

In [ ]:
# --- Variant B: VOTING — same judgment, 3 opinions, majority wins ---
ANGRY = 'Is this customer message angry? Reply exactly YES or NO.\nMessage: "{m}"'
msg = "third time asking. where. is. my. order."

CALLS["n"] = 0
with ThreadPoolExecutor(max_workers=3) as ex:
    votes = list(ex.map(lambda _: ask(ANGRY.format(m=msg), temperature=1.0), range(3)))
verdict = "YES" if votes.count("YES") >= 2 else "NO"
print(f"votes: {votes} → majority: {verdict}")
price_tag("voting (3 calls)")

**Sectioning** = independent subtasks, fan out, fan in. **Voting** = reliability from redundancy — remember W2D4's flaky judge? Majority-of-3 is the standard cure (we even set temperature HIGH here so the votes are genuinely independent opinions).

**✏️ Your turn:** make the vote 5-wide and find a message where the vote splits 3–2. Borderline cases are exactly where voting earns its cost.

---
## Pattern 4 — Orchestrator–workers 🧠

The difference from parallelization: **the LLM writes the subtask list.** You don't know the plan until runtime — but execution is still one bounded pass:

In [ ]:
TOOL_DESCS = "\n".join(f"- {n}: {f.__doc__.strip()}" for n, f in TOOLS.items())

PLAN = """You are an orchestrator. Break this task into 3-5 subtasks.
Available tools (a subtask may use one):
{tools}
Product IDs available: P-101, P-102, P-103, P-104.

Return ONLY JSON:
{{"subtasks": [{{"id": 1, "goal": "...", "tool": "get_price" | "check_inventory" | null,
                "args": {{"product_id": "..."}} | null}}]}}

Task: {task}"""

WORKER = """Complete this one subtask. Be concise.
Goal: {goal}
Results so far: {context}"""

SYNTH = """Combine these subtask results into the final deliverable.
Task: {task}
Results: {results}"""

def orchestrate(task, verbose=True):
    plan = json.loads(ask(PLAN.format(tools=TOOL_DESCS, task=task), temperature=0)
                      .removeprefix("```json").removeprefix("```").removesuffix("```").strip())
    if verbose:
        print("PLAN (written by the LLM at runtime):")
        for st in plan["subtasks"]:
            print(f"  {st['id']}. {st['goal']}" + (f"  [tool: {st['tool']}]" if st.get("tool") else ""))
    results = {}
    for st in plan["subtasks"]:
        if st.get("tool") and st["tool"] in TOOLS:
            results[st["id"]] = TOOLS[st["tool"]](**(st.get("args") or {}))
        else:
            results[st["id"]] = ask(WORKER.format(goal=st["goal"], context=json.dumps(results)),
                                    max_tokens=150)
    return ask(SYNTH.format(task=task, results=json.dumps(results)), max_tokens=350)

CALLS["n"] = 0
print(orchestrate("Write a short kettle buying guide for the SmartMart website comparing our budget and premium options, with live prices and stock."))
price_tag("orchestrator")

**Read the printed plan** — the LLM chose the subtasks, and some carry `tool:` tags: workers reached into yesterday's `TOOLS` registry. Patterns compose across days.

**Cost check:** the counter shows `plan(1) + LLM-workers(k) + synth(1)` — tool workers were free. You knew the price the moment the plan printed, *before* the work ran. That's the orchestrator's promise: dynamic decomposition, bounded execution.

**✏️ Your turn:** run `orchestrate("plan a chai stall for the college fest")` — a task with NO relevant tools. Does the plan sensibly use zero tools? That's the orchestrator degrading gracefully.

---
## Pattern 5 — Evaluator–optimizer 🔁

W2D4's judge, promoted from *grading* to *steering*. The generator drafts; the evaluator returns a verdict **plus concrete feedback**; loop until PASS (bounded!):

In [ ]:
GENERATE = "Write a 2-sentence WhatsApp promo for SmartMart's 10% kitchen sale ending Sunday. Audience: Pune families."
EVALUATE = """You are a strict marketing editor. Check this promo:
1. Under 40 words?  2. Mentions the 10% discount AND Sunday deadline?  3. No spammy ALL-CAPS or excessive punctuation?
Return ONLY JSON: {{"pass": true/false, "feedback": "<one concrete fix if failing>"}}
Promo: {draft}"""
REVISE = "Revise this promo applying the feedback exactly. Feedback: {fb}\nPromo: {draft}"

CALLS["n"] = 0
draft = ask(GENERATE, temperature=1.1)          # deliberately hot — more likely to fail checks
print(f"DRAFT 0: {draft}\n")

for round_ in range(1, 4):                       # bounded: max 3 rounds
    v = json.loads(ask(EVALUATE.format(draft=draft), temperature=0)
                   .removeprefix("```json").removeprefix("```").removesuffix("```").strip())
    if v["pass"]:
        print(f"ROUND {round_}: ✅ PASS")
        break
    print(f"ROUND {round_}: ❌ {v['feedback']}")
    draft = ask(REVISE.format(fb=v["feedback"], draft=draft), temperature=0.5)
    print(f"DRAFT {round_}: {draft}\n")

print(f"\nSHIP IT → {draft}")
price_tag("evaluator loop")

**Two design choices worth stealing:** the evaluator returns *actionable feedback*, not just a score — "mention the deadline" beats "6/10". And the generator runs HOT (creative) while the evaluator runs at temp 0 (strict) — different jobs, different dials (W1D4).

**✏️ Your turn:** add check #4 to the evaluator — "must include a call to action" — and watch a previously-passing draft fail and recover.

---
## 🏁 Finale — the SmartSupport pipeline

Routing + chaining + evaluation, composed. Worst-case cost computable on paper: classify(1) + handle(≤2) + evaluate(≤2×2) = **≤ 7 calls**, guaranteed:

In [ ]:
REPLY_EVAL = """Check this customer-service reply:
1. Correct tone for the situation?  2. No invented policies (only: 7-day returns w/ receipt, store credit without)?
3. Under 4 sentences?
Return ONLY JSON: {{"pass": true/false, "feedback": "<one concrete fix>"}}
Customer message: {msg}
Reply: {reply}"""

class SmartSupport:
    """route → specialist handling (complaint gets a 2-step chain) → evaluator gate."""

    def handle(self, msg, verbose=True):
        start = CALLS["n"]
        # 1 · ROUTE
        route = ask(CLASSIFY.format(msg=msg), temperature=0)
        route = route if route in ROUTES else "OTHER"
        if verbose: print(f"  ROUTE     → {route}")
        # 2 · HANDLE (complaints get a chain: extract facts → draft with facts)
        if route == "COMPLAINT":
            facts = ask(f'List the customer\'s specific problems as bullets, nothing else: "{msg}"', temperature=0)
            if verbose: print(f"  CHAIN 1/2 → facts extracted")
            reply = ask(ROUTES[route].format(msg=msg) + f"\nTheir specific issues:\n{facts}", temperature=0.6)
            if verbose: print(f"  CHAIN 2/2 → draft ready")
        else:
            reply = ask(ROUTES[route].format(msg=msg), temperature=0.6)
        # 3 · EVALUATE (max 2 rounds)
        for round_ in range(2):
            v = json.loads(ask(REPLY_EVAL.format(msg=msg, reply=reply), temperature=0)
                           .removeprefix("```json").removeprefix("```").removesuffix("```").strip())
            if v["pass"]:
                if verbose: print(f"  EVALUATE  → ✅ pass")
                break
            if verbose: print(f"  EVALUATE  → ❌ {v['feedback']}")
            reply = ask(f"Fix this reply. Feedback: {v['feedback']}\nReply: {reply}", temperature=0.4)
        if verbose: print(f"  COST      → {CALLS['n'] - start} calls (≤7 guaranteed)")
        return reply

CALLS["n"] = 0
pipe = SmartSupport()
for msg in ["kettle broke AGAIN, second replacement, and your store hung up on me. i want answers.",
            "what's your return window if I lost my bill?"]:
    print(f"🧑 {msg}")
    print(f"🛒 {pipe.handle(msg)}")
    print("-" * 72)

**No unbounded loops anywhere** — and it still routed, chained, and self-checked. This composed workflow is what most production "AI agents" actually are.

**✏️ Your turn:** ask the same complaint twice. The route and cost stay stable even when wording varies — compare with yesterday's agent traces. THAT's the control you're buying.

---
## 🏆 Homework (20 min, feeds Friday's project)

1. **Pattern-match 5 tasks** — for each, name the pattern and justify in one line: (a) translate 200 reviews to English, (b) turn a support email into a Jira-style ticket JSON, (c) "find me the best kettle under my budget and check it's deliverable to my pin code", (d) generate a poem that must pass 4 style rules, (e) answer FAQ vs complaint vs order-status messages.
2. **Parallel-ize the chain** — in the Pattern 1 pipeline, EXTRACT and a new `SEO_KEYWORDS` prompt don't depend on each other. Run them concurrently; prove the latency win with timestamps.
3. **Bound-check** — compute (on paper) the worst-case call count of your SmartSupport pipeline if the evaluator max rounds were 3 and complaints got a 3-step chain. This exact skill — pricing a design before building it — appears on Friday's rubric.

### What's next
**Module 3 — the loop, done properly:** native function calling (the API's typed version of yesterday's hand-rolled JSON), explicit ReAct reasoning, real memory management, and LangChain — which, after these two days, you'll read like an X-ray.

---
*PMS RoBoTics Research Center · pmsroboticsrc@gmail.com · (+91) 9960664674*